In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')

print('Data source import complete.')

In [ ]:
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Time Series
import warnings
warnings.filterwarnings("ignore")

# Display config
pd.set_option('display.max_columns', None)


In [ ]:
import os

# Load train data and parse date
df = pd.read_csv(os.path.join(store_sales_time_series_forecasting_path, 'train.csv'), parse_dates=['date'])

# Subset for Store 1 and Family 'GROCERY I'
subset_df = df[(df['store_nbr'] == 1) & (df['family'] == 'GROCERY I')].copy()
subset_df.sort_values('date', inplace=True)
subset_df.reset_index(drop=True, inplace=True)

# Overview
print(f"Subset shape: {subset_df.shape}")
subset_df.head()

In [ ]:
# Make sure 'date' is datetime
subset_df['date'] = pd.to_datetime(subset_df['date'])

# Set date as index
subset_df.set_index('date', inplace=True)

# Daily Sales Trend
plt.figure(figsize=(14, 4))
subset_df['sales'].plot()
plt.title('Daily Sales Trend (Store 1 - GROCERY I)')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True)
plt.show()


In [ ]:
# Monthly average sales
monthly = subset_df['sales'].resample('M').mean()

plt.figure(figsize=(12, 4))
monthly.plot(marker='o')
plt.title('Monthly Average Sales')
plt.ylabel('Sales')
plt.grid(True)
plt.show()


In [ ]:
# Add weekday column (0=Monday, 6=Sunday)
subset_df['weekday'] = subset_df.index.weekday

# Plot average sales by weekday
plt.figure(figsize=(8, 4))
sns.barplot(x='weekday', y='sales', data=subset_df)
plt.title('Average Sales by Weekday')
plt.xlabel('Weekday (0 = Monday)')
plt.ylabel('Avg Sales')
plt.grid(True)
plt.show()


In [ ]:
# Compare sales when promotion is on vs off
plt.figure(figsize=(6, 4))
sns.boxplot(x='onpromotion', y='sales', data=subset_df)
plt.title('Sales Distribution: Promotion vs No Promotion')
plt.xlabel('On Promotion')
plt.ylabel('Sales')
plt.grid(True)
plt.show()


In [ ]:
# 30-day rolling mean
subset_df['sales'].rolling(window=30).mean().plot(figsize=(14, 4), label='30-day Moving Avg')
subset_df['sales'].plot(alpha=0.4, label='Actual Sales')
plt.title('Sales with 30-day Moving Average')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(subset_df.index, subset_df['sales'], color='teal', linewidth=1.2)
plt.title('Daily Sales Trend — Store 1 | GROCERY I', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
monthly_avg = subset_df['sales'].resample('M').mean()

plt.figure(figsize=(12, 4))
plt.plot(monthly_avg.index, monthly_avg.values, marker='o', color='darkorange')
plt.title('Monthly Average Sales', fontsize=13)
plt.xlabel('Month')
plt.ylabel('Avg Sales')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
weekday_avg = subset_df.groupby(subset_df.index.day_name())['sales'].mean().reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

plt.figure(figsize=(10, 4))
sns.barplot(x=weekday_avg.index, y=weekday_avg.values, palette='coolwarm')
plt.title('Average Sales by Day of Week', fontsize=13)
plt.ylabel('Avg Sales')
plt.xticks(rotation=45)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='onpromotion', y='sales', data=subset_df, palette='Set2')
plt.title('Sales Distribution: Promotion vs No Promotion', fontsize=12)
plt.xticks([0, 1], ['No', 'Yes'])
plt.xlabel('On Promotion')
plt.ylabel('Sales')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Optional: Load holiday data
holidays = pd.read_csv(os.path.join(store_sales_time_series_forecasting_path, 'holidays_events.csv'), parse_dates=['date'])

# Merge holiday info
holiday_merge = pd.merge(
    subset_df.reset_index(),
    holidays[['date', 'type']],
    how='left',
    on='date'
)

# Add holiday flag
holiday_merge['is_holiday'] = holiday_merge['type'].notna().astype(int)

# Compare sales on holiday vs non-holiday
plt.figure(figsize=(6, 4))
sns.boxplot(x='is_holiday', y='sales', data=holiday_merge, palette='Paired')
plt.title('Sales on Holidays vs Non-Holidays')
plt.xticks([0, 1], ['Non-Holiday', 'Holiday'])
plt.ylabel('Sales')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 4))
plot_acf(subset_df['sales'], lags=30)
plt.title('Autocorrelation Plot (Sales)')
plt.tight_layout()
plt.show()


In [ ]:
# Extract year from date
subset_df['year'] = subset_df.index.year

# Group and plot
yearly_avg = subset_df.groupby('year')['sales'].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=yearly_avg.index.astype(str), y=yearly_avg.values, palette='viridis')
plt.title('Average Sales Per Year')
plt.xlabel('Year')
plt.ylabel('Avg Sales')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
subset_df['month'] = subset_df.index.month

monthly_avg = subset_df.groupby('month')['sales'].mean()

plt.figure(figsize=(10, 4))
sns.lineplot(x=monthly_avg.index, y=monthly_avg.values, marker='o', color='darkgreen')
plt.title('Average Sales by Month (All Years Combined)')
plt.xlabel('Month')
plt.ylabel('Avg Sales')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
subset_df['day'] = subset_df.index.day

day_avg = subset_df.groupby('day')['sales'].mean()

plt.figure(figsize=(12, 4))
sns.lineplot(x=day_avg.index, y=day_avg.values, color='purple')
plt.title('Average Sales by Day of Month')
plt.xlabel('Day')
plt.ylabel('Avg Sales')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Ensure daily frequency
ts = subset_df['sales'].asfreq('D')

# Fill missing values
ts = ts.fillna(method='ffill')

# Decompose
decomp = seasonal_decompose(ts, model='additive', period=365)

# Plot
decomp.plot()
plt.suptitle('Seasonal Decomposition (Additive)', fontsize=14)
plt.tight_layout()
plt.show()

missing_days = ts[ts.isna()]
print(f"Missing days count: {missing_days.shape[0]}")

In [ ]:
# Reset index if needed
data = subset_df[['sales']].copy()
data = data.asfreq('D')  # ensure daily frequency
data = data.fillna(method='ffill')

# Set cutoff date (e.g., last 3 months for testing)
split_date = '2017-07-01'

train = data.loc[:split_date].copy()
test = data.loc[split_date:].copy()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")


In [ ]:
train['log_sales'] = np.log1p(train['sales'])  # log(1 + x)
test['log_sales'] = np.log1p(test['sales'])


In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(train['log_sales'].dropna())
print("ADF Statistic:", result[0])
print("p-value:", result[1])
for key, value in result[4].items():
    print(f"Critical Value ({key}): {value}")

if result[1] < 0.05:
    print("✅ The series is stationary.")
else:
    print("⚠️ The series is non-stationary.")


In [ ]:
rol_mean = train['log_sales'].rolling(window=30).mean()
rol_std = train['log_sales'].rolling(window=30).std()

plt.figure(figsize=(12, 5))
plt.plot(train['log_sales'], color='blue', label='Log Transformed Sales')
plt.plot(rol_mean, color='red', label='Rolling Mean (30)')
plt.plot(rol_std, color='black', label='Rolling Std (30)')
plt.legend(loc='best')
plt.title('Rolling Mean & Standard Deviation (Log Sales)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Fit ARIMA on log-transformed data
model_arima = ARIMA(train['log_sales'], order=(5,1,0))
model_arima_fit = model_arima.fit()

# Forecast steps ahead
forecast_steps = len(test)
forecast_log = model_arima_fit.forecast(steps=forecast_steps)
forecast_arima = np.expm1(forecast_log)  # convert back from log

# Evaluation
rmse_arima = np.sqrt(mean_squared_error(test['sales'], forecast_arima))
mae_arima = mean_absolute_error(test['sales'], forecast_arima)

print("ARIMA RMSE:", rmse_arima)
print("ARIMA MAE:", mae_arima)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train.index, train['sales'], label='Train')
plt.plot(test.index, test['sales'], label='Actual')
plt.plot(test.index, forecast_arima, label='ARIMA Forecast', color='green')
plt.title('ARIMA Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from prophet import Prophet

# Prophet needs 'ds' and 'y'
prophet_df = subset_df.reset_index()[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})

# Filter again for same subset
prophet_df = prophet_df[(prophet_df['ds'] < '2017-08-01') & (prophet_df['ds'] >= '2013-01-01')]

# Split
train_prophet = prophet_df[prophet_df['ds'] < '2017-07-01']
test_prophet = prophet_df[prophet_df['ds'] >= '2017-07-01']


In [ ]:
m = Prophet()
m.fit(train_prophet)

future = m.make_future_dataframe(periods=len(test_prophet))
forecast = m.predict(future)

# Evaluation
y_true = test_prophet['y'].values
y_pred = forecast.iloc[-len(test_prophet):]['yhat'].values

rmse_prophet = np.sqrt(mean_squared_error(y_true, y_pred))
mae_prophet = mean_absolute_error(y_true, y_pred)

print("Prophet RMSE:", rmse_prophet)
print("Prophet MAE:", mae_prophet)


In [ ]:
fig = m.plot(forecast)
plt.title('Prophet Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()


In [ ]:
df_ml = subset_df[['sales']].copy()
df_ml['day'] = df_ml.index.day
df_ml['month'] = df_ml.index.month
df_ml['weekday'] = df_ml.index.weekday
df_ml['year'] = df_ml.index.year

# Lag features
df_ml['lag_1'] = df_ml['sales'].shift(1)
df_ml['lag_7'] = df_ml['sales'].shift(7)
df_ml['rolling_mean_7'] = df_ml['sales'].shift(1).rolling(window=7).mean()

# Drop NaNs
df_ml.dropna(inplace=True)


In [ ]:
X = df_ml.drop('sales', axis=1)
y = df_ml['sales']

# Split index (same as previous)
split_date = '2017-07-01'
X_train = X.loc[:split_date]
y_train = y.loc[:split_date]
X_test = X.loc[split_date:]
y_test = y.loc[split_date:]


In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = model_xgb.predict(X_test)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost MAE:", mae_xgb)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test, label='Actual Sales')
plt.plot(y_test.index, y_pred_xgb, label='XGBoost Forecast', color='orange')
plt.title('XGBoost Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

# Calculate MAPE for Prophet and XGBoost
mape_arima = mean_absolute_percentage_error(test['sales'], forecast_arima)
mape_prophet = mean_absolute_percentage_error(y_true, y_pred)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)

# Final comparison
results = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'XGBoost'],
    'RMSE': [rmse_arima, rmse_prophet, rmse_xgb],
    'MAE': [mae_arima, mae_prophet, mae_xgb],
    'MAPE': [mape_arima, mape_prophet, mape_xgb]
})

results.sort_values('RMSE')


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test, label='Actual Sales')
plt.plot(y_test.index, y_pred_xgb, label='XGBoost Forecast', color='orange')
plt.title('XGBoost Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(x='Model', y='RMSE', data=results, palette='Set2')
plt.title('Model Comparison: RMSE')
plt.ylabel('RMSE')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(test.index, test['sales'], label='Actual', color='black')
plt.plot(test.index, forecast_arima, label='ARIMA', linestyle='--')
plt.plot(test_prophet['ds'], y_pred, label='Prophet', linestyle='--')
plt.plot(y_test.index, y_pred_xgb, label='XGBoost', linestyle='--')
plt.title('Forecast Comparison')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
